# Lab Proposal Notebook Experiment

This notebook follows `Cambagent_lab_proposal.pdf` and gives you a notebook-first way to run and inspect the scaffolded experiment matrix.

Use it to:
- load the baseline config
- clone and edit the experiment matrix inline
- run the offline scaffold
- inspect metrics and failure modes without depending on the CLI


In [ ]:
from pathlib import Path
import os

def resolve_repo_root() -> Path:
    candidates: list[Path] = []
    env_root = os.getenv("CAMBAGENT_EVAL_REPO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path(r"C:\Users\Anwender\Science-Work-Flow-"))

    seen: set[str] = set()
    for candidate in candidates:
        resolved = candidate.resolve()
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)
        if (resolved / "pyproject.toml").exists() and (resolved / "src").is_dir():
            return resolved

    raise RuntimeError(
        "Repo root not found. Set CAMBAGENT_EVAL_REPO_ROOT or open the notebook from the repo checkout."
    )

ROOT = resolve_repo_root()
ROOT

In [ ]:
from pathlib import Path
import sys

EXPECTED_KERNEL_NAME = 'pytorch-gpu-311'
EXPECTED_PYTHON = Path('C:\\Users\\Anwender\\AppData\\Local\\anaconda3\\envs\\pytorch-gpu\\python.exe')
CURRENT_PYTHON = Path(sys.executable).resolve()
KERNEL_STATUS = {
    'expected_kernel_name': EXPECTED_KERNEL_NAME,
    'expected_python': str(EXPECTED_PYTHON),
    'current_python': str(CURRENT_PYTHON),
    'kernel_ok': CURRENT_PYTHON == EXPECTED_PYTHON,
}
if not KERNEL_STATUS['kernel_ok']:
    print('Warning: this notebook is not using the recommended local Python kernel.')
KERNEL_STATUS


In [ ]:
from collections import Counter, defaultdict
from copy import deepcopy
from dataclasses import asdict, dataclass, field
from datetime import UTC, datetime
import hashlib
import json
from typing import Any, Callable

In [ ]:
@dataclass(slots=True)
class StressCondition:
    id: str
    description: str = ""
    severity: str = "medium"
    applies_to: list[str] = field(default_factory=list)
    effect_size: float = 0.0

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "StressCondition":
        return cls(
            id=str(data.get("id", data.get("name", "unnamed_stress"))),
            description=str(data.get("description", "")),
            severity=str(data.get("severity", "medium")),
            applies_to=[str(item) for item in data.get("applies_to", [])],
            effect_size=float(data.get("effect_size", 0.0)),
        )


@dataclass(slots=True)
class TaskCase:
    id: str
    name: str
    task_type: str
    default_workflow: str
    supported_workflows: list[str]
    prompt: str
    ground_truth: dict[str, float]
    expected_tools: list[str] = field(default_factory=list)
    domain_context: list[str] = field(default_factory=list)
    priors: dict[str, list[float]] = field(default_factory=dict)
    metadata: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "TaskCase":
        default_workflow = str(data["default_workflow"])
        supported_workflows = [str(item) for item in data.get("supported_workflows", [default_workflow])]
        ground_truth = {str(key): float(value) for key, value in data["ground_truth"].items()}
        priors = {
            str(key): [float(bound) for bound in value]
            for key, value in data.get("priors", {}).items()
        }
        return cls(
            id=str(data["id"]),
            name=str(data["name"]),
            task_type=str(data["task_type"]),
            default_workflow=default_workflow,
            supported_workflows=supported_workflows,
            prompt=str(data["prompt"]),
            ground_truth=ground_truth,
            expected_tools=[str(item) for item in data.get("expected_tools", [])],
            domain_context=[str(item) for item in data.get("domain_context", [])],
            priors=priors,
            metadata=dict(data.get("metadata", {})),
        )


@dataclass(slots=True)
class ExperimentSpec:
    id: str
    name: str
    description: str
    task_files: list[str]
    models: list[str]
    grounding_modes: list[str]
    workflows: list[str]
    stress_tests: list[StressCondition] = field(default_factory=list)
    report_path: str = "outputs/report.json"
    source_path: str = ""

    @classmethod
    def from_dict(cls, data: dict[str, Any]) -> "ExperimentSpec":
        return cls(
            id=str(data["id"]),
            name=str(data.get("name", data["id"])),
            description=str(data.get("description", "")),
            task_files=[str(item) for item in data.get("task_files", [])],
            models=[str(item) for item in data.get("models", [])],
            grounding_modes=[str(item) for item in data.get("grounding_modes", [])],
            workflows=[str(item) for item in data.get("workflows", [])],
            stress_tests=[StressCondition.from_dict(item) for item in data.get("stress_tests", [])],
            report_path=str(data.get("report_path", "outputs/report.json")),
        )


@dataclass(slots=True)
class TraceStep:
    index: int
    kind: str
    summary: str
    tool_name: str | None = None
    notes: list[str] = field(default_factory=list)


@dataclass(slots=True)
class RunResult:
    task_id: str
    task_name: str
    task_type: str
    model_name: str
    grounding_mode: str
    workflow: str
    stress_test: str
    metrics: dict[str, float]
    failures: list[str]
    final_output: dict[str, Any]
    steps: list[TraceStep]

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass(slots=True)
class ExperimentReport:
    experiment_id: str
    experiment_name: str
    generated_at: str
    source_config: str
    summary: dict[str, Any]
    runs: list[RunResult]

    def to_dict(self) -> dict[str, Any]:
        return {
            "experiment_id": self.experiment_id,
            "experiment_name": self.experiment_name,
            "generated_at": self.generated_at,
            "source_config": self.source_config,
            "summary": self.summary,
            "runs": [run.to_dict() for run in self.runs],
        }

In [ ]:
def _load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def _resolve_relative_path(base_dir: Path, candidate: str) -> Path:
    path = Path(candidate)
    if path.is_absolute():
        return path
    return (base_dir / path).resolve()


def load_experiment(path: str | Path) -> ExperimentSpec:
    experiment_path = Path(path).resolve()
    spec = ExperimentSpec.from_dict(_load_json(experiment_path))
    spec.source_path = str(experiment_path)
    spec.task_files = [
        str(_resolve_relative_path(experiment_path.parent, task_file))
        for task_file in spec.task_files
    ]
    spec.report_path = str(_resolve_relative_path(experiment_path.parent, spec.report_path))
    return spec


def load_tasks(spec: ExperimentSpec) -> list[TaskCase]:
    return [TaskCase.from_dict(_load_json(Path(task_file))) for task_file in spec.task_files]


def validate_experiment(spec: ExperimentSpec, tasks: list[TaskCase]) -> list[str]:
    problems: list[str] = []
    if not spec.models:
        problems.append("Experiment is missing models.")
    if not spec.grounding_modes:
        problems.append("Experiment is missing grounding modes.")
    if not spec.workflows:
        problems.append("Experiment is missing workflows.")
    if not tasks:
        problems.append("Experiment does not reference any task files.")

    for task in tasks:
        if task.default_workflow not in task.supported_workflows:
            problems.append(
                f"Task '{task.id}' default workflow '{task.default_workflow}' is not supported."
            )
        if not set(task.supported_workflows).intersection(spec.workflows):
            problems.append(
                f"Task '{task.id}' has no workflow overlap with experiment workflows."
            )
        if set(task.priors) != set(task.ground_truth):
            problems.append(
                f"Task '{task.id}' priors do not cover the same parameters as ground truth."
            )
    return problems

In [ ]:
def build_trace(
    workflow_name: str,
    task: TaskCase,
    used_tools: list[str],
    grounding_mode: str,
    stress_test: StressCondition,
    confidence: float,
) -> list[TraceStep]:
    if workflow_name == "deep_research":
        retrieval_note = (
            "Domain notes were incorporated before inference."
            if grounding_mode != "plain"
            else "No external grounding was injected."
        )
        inference_tool = used_tools[1] if len(used_tools) > 1 else (used_tools[0] if used_tools else None)
        return [
            TraceStep(
                index=1,
                kind="plan",
                summary="Break the research task into planning, retrieval, inference, and synthesis stages.",
                notes=[f"Stress test: {stress_test.id}"],
            ),
            TraceStep(
                index=2,
                kind="retrieve",
                summary="Gather domain priors or literature cues that constrain the analysis.",
                tool_name=used_tools[0] if used_tools else None,
                notes=[retrieval_note],
            ),
            TraceStep(
                index=3,
                kind="infer",
                summary="Run the inference chain and inspect uncertainty-sensitive outputs.",
                tool_name=inference_tool,
                notes=[f"Domain context items available: {len(task.domain_context)}"],
            ),
            TraceStep(
                index=4,
                kind="synthesize",
                summary="Explain the posterior in natural language and flag potential failure modes.",
                notes=[f"Reported confidence: {confidence:.3f}"],
            ),
        ]

    tool_name = used_tools[0] if used_tools else None
    notes = [
        f"Grounding mode: {grounding_mode}",
        f"Stress test: {stress_test.id}",
        f"Expected tools: {', '.join(task.expected_tools) or 'none'}",
    ]
    return [
        TraceStep(
            index=1,
            kind="interpret",
            summary="Parse the precision task and map requested outputs to solver parameters.",
            notes=notes,
        ),
        TraceStep(
            index=2,
            kind="tool_call",
            summary="Execute the primary scientific computation.",
            tool_name=tool_name,
            notes=[f"Domain context items available: {len(task.domain_context)}"],
        ),
        TraceStep(
            index=3,
            kind="report",
            summary="Return parameter estimates with a compact confidence statement.",
            notes=[f"Reported confidence: {confidence:.3f}"],
        ),
    ]


class StubScientificAgent:
    def run(
        self,
        task: TaskCase,
        model_name: str,
        grounding_mode: str,
        workflow_name: str,
        stress_test: StressCondition,
    ) -> tuple[list[TraceStep], dict[str, Any]]:
        quality = self._estimate_quality(task, model_name, grounding_mode, workflow_name, stress_test)
        used_tools = self._select_tools(task, quality, grounding_mode, workflow_name)
        parameters = self._predict_parameters(task, model_name, grounding_mode, workflow_name, stress_test, quality)
        confidence = max(0.05, min(0.99, quality + 0.08 * self._signed_noise(task.id, model_name, "confidence")))
        steps = build_trace(workflow_name, task, used_tools, grounding_mode, stress_test, confidence)

        final_output = {
            "parameters": parameters,
            "used_tools": used_tools,
            "confidence": round(confidence, 3),
            "summary": self._summary_text(task, workflow_name, grounding_mode, stress_test),
        }
        return steps, final_output

    def _estimate_quality(
        self,
        task: TaskCase,
        model_name: str,
        grounding_mode: str,
        workflow_name: str,
        stress_test: StressCondition,
    ) -> float:
        quality = 0.58
        model_name_lower = model_name.lower()
        if "frontier" in model_name_lower or "large" in model_name_lower:
            quality += 0.18
        if grounding_mode != "plain":
            quality += 0.10
        if workflow_name == task.default_workflow:
            quality += 0.04
        quality -= stress_test.effect_size
        quality += 0.05 * self._signed_noise(task.id, model_name, grounding_mode, workflow_name, stress_test.id)
        return max(0.08, min(0.97, quality))

    def _predict_parameters(
        self,
        task: TaskCase,
        model_name: str,
        grounding_mode: str,
        workflow_name: str,
        stress_test: StressCondition,
        quality: float,
    ) -> dict[str, float]:
        parameters: dict[str, float] = {}
        error_scale = max(0.015, 0.32 - (quality * 0.26) + (stress_test.effect_size * 0.35))
        for key, truth in task.ground_truth.items():
            noise = self._signed_noise(task.id, model_name, grounding_mode, workflow_name, stress_test.id, key)
            predicted = truth * (1 + noise * error_scale)
            if stress_test.id == "missing_data" and task.task_type == "research_inference" and key == "eccentricity":
                predicted += 0.04
            if stress_test.id == "parameter_misconfiguration" and task.task_type == "tool_precision" and key == "omega_b":
                predicted -= 0.006
            parameters[key] = round(predicted, 6)
        return parameters

    def _select_tools(
        self,
        task: TaskCase,
        quality: float,
        grounding_mode: str,
        workflow_name: str,
    ) -> list[str]:
        tools = list(task.expected_tools)
        if not tools:
            return tools

        if grounding_mode == "plain" and quality < 0.65:
            tools[-1] = "hallucinated_solver"
        elif grounding_mode != "plain" and workflow_name == "deep_research" and "paper_index" not in tools:
            tools.insert(0, "paper_index")
        return tools

    def _summary_text(
        self,
        task: TaskCase,
        workflow_name: str,
        grounding_mode: str,
        stress_test: StressCondition,
    ) -> str:
        return (
            f"Simulated {workflow_name} run for '{task.name}' with "
            f"{grounding_mode} grounding under the '{stress_test.id}' condition."
        )

    def _signed_noise(self, *parts: str) -> float:
        digest = hashlib.sha256("|".join(parts).encode("utf-8")).hexdigest()
        bucket = int(digest[:8], 16) / 0xFFFFFFFF
        return (bucket * 2.0) - 1.0


def _within_priors(task: TaskCase, predicted: dict[str, float]) -> bool:
    for key, bounds in task.priors.items():
        value = float(predicted.get(key, 0.0))
        lower, upper = bounds
        if value < lower or value > upper:
            return False
    return True


def compute_metrics(
    task: TaskCase,
    final_output: dict,
    workflow_name: str,
    stress_test: StressCondition,
) -> dict[str, float]:
    predicted = final_output.get("parameters", {})
    expected_tools = set(task.expected_tools)
    used_tools = set(final_output.get("used_tools", []))
    confidence = float(final_output.get("confidence", 0.5))

    relative_errors: list[float] = []
    for key, truth in task.ground_truth.items():
        predicted_value = float(predicted.get(key, 0.0))
        denominator = max(abs(truth), 1e-9)
        relative_errors.append(abs(predicted_value - truth) / denominator)

    mean_relative_error = sum(relative_errors) / len(relative_errors)
    max_relative_error = max(relative_errors)

    union = expected_tools | used_tools
    tool_use_accuracy = 1.0 if not union else len(expected_tools & used_tools) / len(union)
    hallucinated_tool_calls = float(len(used_tools - expected_tools))
    physical_validity = 1.0 if _within_priors(task, predicted) else 0.0

    numerical_accuracy = max(0.0, 1.0 - mean_relative_error)
    posterior_consistency = max(0.0, 1.0 - ((mean_relative_error + max_relative_error) / 2.0))
    stability_under_perturbation = max(0.0, numerical_accuracy - stress_test.effect_size)
    error_propagation_risk = min(
        1.0,
        max_relative_error * (1.2 if workflow_name == "deep_research" else 0.8),
    )
    confidence_calibration_gap = abs(confidence - numerical_accuracy)

    return {
        "mean_relative_error": round(mean_relative_error, 6),
        "max_relative_error": round(max_relative_error, 6),
        "numerical_accuracy": round(numerical_accuracy, 6),
        "posterior_consistency": round(posterior_consistency, 6),
        "tool_use_accuracy": round(tool_use_accuracy, 6),
        "hallucinated_tool_calls": round(hallucinated_tool_calls, 6),
        "physical_validity": round(physical_validity, 6),
        "stability_under_perturbation": round(stability_under_perturbation, 6),
        "error_propagation_risk": round(error_propagation_risk, 6),
        "confidence_calibration_gap": round(confidence_calibration_gap, 6),
    }


def classify_failures(task: TaskCase, workflow_name: str, metrics: dict[str, float]) -> list[str]:
    failures: list[str] = []

    if metrics["hallucinated_tool_calls"] > 0:
        failures.append("hallucinated_tool_call")
    if metrics["physical_validity"] < 1.0:
        failures.append("physically_inconsistent_output")
    if metrics["numerical_accuracy"] < 0.9 and metrics["hallucinated_tool_calls"] == 0:
        failures.append("silent_numerical_error")
    if metrics["confidence_calibration_gap"] > 0.2:
        failures.append("reasoning_output_mismatch")
    if workflow_name == "deep_research" and (
        metrics["error_propagation_risk"] > 0.08 or len(failures) >= 2
    ):
        failures.append("error_propagation")

    if not failures:
        failures.append("no_detected_failure")
    return failures


def _stress_plan(spec: ExperimentSpec, task: TaskCase) -> list[StressCondition]:
    baseline = StressCondition(id="baseline", description="No perturbation.", severity="none", effect_size=0.0)
    applicable = [baseline]
    for stress_test in spec.stress_tests:
        if not stress_test.applies_to or task.task_type in stress_test.applies_to:
            applicable.append(stress_test)
    return applicable


def _group_metrics(
    runs: list[RunResult],
    key_fn: Callable[[RunResult], str],
) -> dict[str, dict[str, float]]:
    grouped: dict[str, list[RunResult]] = defaultdict(list)
    for run in runs:
        grouped[key_fn(run)].append(run)

    output: dict[str, dict[str, float]] = {}
    for group_name, group_runs in grouped.items():
        metric_names = group_runs[0].metrics.keys()
        output[group_name] = {
            "runs": float(len(group_runs)),
        }
        for metric_name in metric_names:
            output[group_name][metric_name] = round(
                sum(run.metrics[metric_name] for run in group_runs) / len(group_runs),
                6,
            )
    return dict(sorted(output.items()))


def summarize_runs(runs: list[RunResult]) -> dict:
    metric_names = list(runs[0].metrics.keys()) if runs else []
    average_metrics = {
        metric_name: round(sum(run.metrics[metric_name] for run in runs) / len(runs), 6)
        for metric_name in metric_names
    } if runs else {}

    failure_counts = Counter()
    for run in runs:
        failure_counts.update(run.failures)

    return {
        "total_runs": len(runs),
        "average_metrics": average_metrics,
        "by_model": _group_metrics(runs, lambda run: run.model_name),
        "by_grounding_mode": _group_metrics(runs, lambda run: run.grounding_mode),
        "by_workflow": _group_metrics(runs, lambda run: run.workflow),
        "by_stress_test": _group_metrics(runs, lambda run: run.stress_test),
        "failure_counts": dict(sorted(failure_counts.items())),
    }


def run_experiment(
    spec: ExperimentSpec,
    tasks: list[TaskCase],
    agent: Any | None = None,
) -> ExperimentReport:
    agent = agent or StubScientificAgent()
    runs: list[RunResult] = []

    for task in tasks:
        workflows = [workflow for workflow in spec.workflows if workflow in task.supported_workflows]
        stress_tests = _stress_plan(spec, task)
        for workflow_name in workflows:
            for model_name in spec.models:
                for grounding_mode in spec.grounding_modes:
                    for stress_test in stress_tests:
                        steps, final_output = agent.run(
                            task=task,
                            model_name=model_name,
                            grounding_mode=grounding_mode,
                            workflow_name=workflow_name,
                            stress_test=stress_test,
                        )
                        metrics = compute_metrics(task, final_output, workflow_name, stress_test)
                        failures = classify_failures(task, workflow_name, metrics)
                        runs.append(
                            RunResult(
                                task_id=task.id,
                                task_name=task.name,
                                task_type=task.task_type,
                                model_name=model_name,
                                grounding_mode=grounding_mode,
                                workflow=workflow_name,
                                stress_test=stress_test.id,
                                metrics=metrics,
                                failures=failures,
                                final_output=final_output,
                                steps=steps,
                            )
                        )

    summary = summarize_runs(runs)
    return ExperimentReport(
        experiment_id=spec.id,
        experiment_name=spec.name,
        generated_at=datetime.now(UTC).isoformat(timespec="seconds").replace("+00:00", "Z"),
        source_config=spec.source_path,
        summary=summary,
        runs=runs,
    )


def write_report(report: ExperimentReport, path: str | Path) -> Path:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report.to_dict(), indent=2), encoding="utf-8")
    return output_path


def format_summary(report: ExperimentReport) -> str:
    lines = [
        f"Experiment: {report.experiment_name}",
        f"Runs: {report.summary['total_runs']}",
        "Average metrics:",
    ]
    for metric_name, value in report.summary["average_metrics"].items():
        lines.append(f"  - {metric_name}: {value:.3f}")
    lines.append("Failure counts:")
    for failure_name, count in report.summary["failure_counts"].items():
        lines.append(f"  - {failure_name}: {count}")
    return "\n".join(lines)

In [ ]:
def _format_problems(problems: list[str]) -> str:
    bullet_list = "\n".join(f"- {problem}" for problem in problems)
    return f"Experiment config is invalid:\n{bullet_list}"


def load_notebook_experiment(path: str | Path) -> tuple[ExperimentSpec, list[TaskCase]]:
    spec = load_experiment(path)
    tasks = load_tasks(spec)
    problems = validate_experiment(spec, tasks)
    if problems:
        raise ValueError(_format_problems(problems))
    return spec, tasks


def clone_experiment(
    spec: ExperimentSpec,
    *,
    experiment_id: str | None = None,
    name: str | None = None,
    description: str | None = None,
    models: list[str] | None = None,
    grounding_modes: list[str] | None = None,
    workflows: list[str] | None = None,
    report_path: str | None = None,
) -> ExperimentSpec:
    cloned = deepcopy(spec)
    if experiment_id is not None:
        cloned.id = experiment_id
    if name is not None:
        cloned.name = name
    if description is not None:
        cloned.description = description
    if models is not None:
        cloned.models = list(models)
    if grounding_modes is not None:
        cloned.grounding_modes = list(grounding_modes)
    if workflows is not None:
        cloned.workflows = list(workflows)
    if report_path is not None:
        cloned.report_path = report_path
    return cloned


def experiment_overview(spec: ExperimentSpec, tasks: list[TaskCase]) -> dict[str, Any]:
    supported_workflows = sorted({workflow for task in tasks for workflow in task.supported_workflows})
    return {
        "experiment_id": spec.id,
        "experiment_name": spec.name,
        "task_ids": [task.id for task in tasks],
        "models": list(spec.models),
        "grounding_modes": list(spec.grounding_modes),
        "workflows": list(spec.workflows),
        "task_supported_workflows": supported_workflows,
        "stress_tests": [stress.id for stress in spec.stress_tests],
        "report_path": spec.report_path,
    }


def run_notebook_experiment(
    spec: ExperimentSpec,
    tasks: list[TaskCase],
    *,
    output_path: str | Path | None = None,
) -> tuple[ExperimentReport, Path]:
    problems = validate_experiment(spec, tasks)
    if problems:
        raise ValueError(_format_problems(problems))

    report = run_experiment(spec, tasks)
    written_path = write_report(report, output_path or spec.report_path)
    return report, written_path


def report_rows(report: ExperimentReport) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for run in report.runs:
        row: dict[str, Any] = {
            "task_id": run.task_id,
            "task_name": run.task_name,
            "task_type": run.task_type,
            "model_name": run.model_name,
            "grounding_mode": run.grounding_mode,
            "workflow": run.workflow,
            "stress_test": run.stress_test,
            "confidence": float(run.final_output.get("confidence", 0.0)),
            "used_tools": ", ".join(str(tool) for tool in run.final_output.get("used_tools", [])),
            "failures": ", ".join(run.failures),
        }
        for metric_name, metric_value in run.metrics.items():
            row[f"metric__{metric_name}"] = metric_value
        for parameter_name, parameter_value in run.final_output.get("parameters", {}).items():
            row[f"parameter__{parameter_name}"] = parameter_value
        rows.append(row)
    return rows


def filter_rows(
    rows: list[dict[str, Any]],
    *,
    task_id: str | None = None,
    workflow: str | None = None,
    grounding_mode: str | None = None,
    stress_test: str | None = None,
    failure: str | None = None,
) -> list[dict[str, Any]]:
    filtered: list[dict[str, Any]] = []
    for row in rows:
        if task_id is not None and row["task_id"] != task_id:
            continue
        if workflow is not None and row["workflow"] != workflow:
            continue
        if grounding_mode is not None and row["grounding_mode"] != grounding_mode:
            continue
        if stress_test is not None and row["stress_test"] != stress_test:
            continue
        if failure is not None and failure not in row["failures"]:
            continue
        filtered.append(row)
    return filtered

## Customize The Experiment Matrix

Clone the baseline before editing it so your notebook runs stay separate from the checked-in config.


In [ ]:
custom_spec = clone_experiment(
    spec,
    experiment_id="lab-proposal-notebook",
    name="Lab Proposal Notebook Experiment",
    description="Notebook-driven experiment derived from the lab proposal.",
    report_path=str(ROOT / "outputs" / "notebooks" / "lab_proposal_notebook_report.json"),
)

# Edit these lists to try your own ablations.
custom_spec.models = ["small-baseline-llm", "frontier-science-llm"]
custom_spec.grounding_modes = ["plain", "retrieval_augmented"]
custom_spec.workflows = ["one_shot", "deep_research"]

experiment_overview(custom_spec, tasks)


In [ ]:
report, report_path = run_notebook_experiment(custom_spec, tasks)
report_path


In [ ]:
print(format_summary(report))


In [ ]:
rows = report_rows(report)

try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    rows[:5]
else:
    df = pd.DataFrame(rows)
    df[
        [
            "task_id",
            "model_name",
            "grounding_mode",
            "workflow",
            "stress_test",
            "metric__numerical_accuracy",
            "metric__tool_use_accuracy",
            "metric__confidence_calibration_gap",
            "failures",
        ]
    ].sort_values(
        ["task_id", "workflow", "model_name", "grounding_mode", "stress_test"]
    ).reset_index(drop=True)


In [ ]:
interesting_rows = filter_rows(rows, failure="error_propagation")
interesting_rows[:5]
